# 02 — Train MicroSplit Model

Trains a MicroSplit LadderVAE model for 5 epochs on the small demo dataset.

**Run `00_datasets.ipynb` and `01_noisemodels.ipynb` first.**  Run `03_predict.ipynb` next.

In [ ]:
# imports
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from pytorch_lightning import Trainer
from careamics.lightning import VAEModule

sys.path.insert(0, '../../../src')
from microsplit_reproducibility.configs.data.JUMP import get_data_configs
from microsplit_reproducibility.configs.factory import (
    create_algorithm_config,
    get_likelihood_config,
    get_loss_config,
    get_model_config,
    get_lr_scheduler_config,
    get_optimizer_config,
    get_training_config,
)
from microsplit_reproducibility.configs.parameters.JUMP import get_microsplit_parameters
from microsplit_reproducibility.datasets.common import create_lazy_datasets
from microsplit_reproducibility.utils.callbacks import get_callbacks

In [ ]:
DEMO_DIR    = Path('./cpg0000_demo')
DATASET_DIR = DEMO_DIR / 'dataset'
NM_DIR      = DATASET_DIR / 'noise_models'
CKPT_DIR    = DATASET_DIR / 'checkpoints'
CHANNELS    = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']

NUM_EPOCHS      = 5
BATCH_SIZE      = 4
TRAIN_GRID_SIZE = 32

for p, name in [(DATASET_DIR, 'dataset'), (NM_DIR, 'noise_models')]:
    if not p.exists():
        raise FileNotFoundError(f'{name} not found at {p} — run earlier notebooks first.')

torch.set_float32_matmul_precision('high')

In [ ]:
train_data_config, _, _ = get_data_configs(
    channel_idx_list=CHANNELS,
    train_grid_size=TRAIN_GRID_SIZE,
)

train_dset, val_dset, _, data_stats = create_lazy_datasets(
    datapath=str(DATASET_DIR),
    channel_names=CHANNELS,
    train_grid_size=TRAIN_GRID_SIZE,
    val_grid_size=32,
    image_size=64,
    multiscale_lowres_count=train_data_config.multiscale_lowres_count,
    val_fraction=0.2,
    test_fraction=0.1,
    cache_size=32,
)

print(f'Train: {train_dset.get_num_frames()} frames | Val: {val_dset.get_num_frames()} frames')

mean_dict, std_dict = train_dset.get_mean_std()
np.savez(
    str(DATASET_DIR / 'training_stats.npz'),
    mean_input  = np.array(mean_dict['input']),
    std_input   = np.array(std_dict['input']),
    mean_target = np.array(mean_dict['target']),
    std_target  = np.array(std_dict['target']),
    max_val     = np.array(train_dset.get_max_val()),
)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=0, pin_memory=False)
train_loader  = DataLoader(train_dset, shuffle=True,  **loader_kwargs)
val_loader    = DataLoader(val_dset,   shuffle=False, **loader_kwargs)

In [ ]:
experiment_params = get_microsplit_parameters(
    nm_path=str(NM_DIR),
    channel_idx_list=CHANNELS,
    batch_size=BATCH_SIZE,
)
experiment_params['data_stats'] = data_stats

gaussian_lik_config, noise_model_config, nm_lik_config = get_likelihood_config(**experiment_params)
training_config = get_training_config(**experiment_params)
training_config.num_epochs = NUM_EPOCHS

experiment_config = create_algorithm_config(
    algorithm=experiment_params['algorithm'],
    loss_config=get_loss_config(**experiment_params),
    model_config=get_model_config(**experiment_params),
    gaussian_lik_config=gaussian_lik_config,
    nm_config=noise_model_config,
    nm_lik_config=nm_lik_config,
    lr_scheduler_config=get_lr_scheduler_config(**experiment_params),
    optimizer_config=get_optimizer_config(**experiment_params),
)

model = VAEModule(algorithm_config=experiment_config)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
CKPT_DIR.mkdir(parents=True, exist_ok=True)
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'

trainer = Trainer(
    max_epochs=NUM_EPOCHS,
    accelerator=accelerator,
    enable_progress_bar=True,
    callbacks=get_callbacks(str(CKPT_DIR)),
    precision=training_config.precision,
    gradient_clip_val=training_config.gradient_clip_val,
    gradient_clip_algorithm=training_config.gradient_clip_algorithm,
    check_val_every_n_epoch=1,
    default_root_dir=str(DATASET_DIR),
)

trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=val_loader)